In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer

In [2]:
df = pd.read_csv('mes_donnees_normalisées.csv', sep=';')
df.columns = df.columns.str.strip()
df.loc[df['Lt'] == 'R', 'Lt'] = 0
df.loc[df['Lt'] == 'G', 'Lt'] = 1

In [3]:
#Préparation des données 

# Séparation des features et de la target
df_ut = df[['P1_reel','Lq_reel','P4','V','E2','P6_reel','E1','E2.1','E2.2','E2.3','E2.4']]
#P5 P2 Lt
p1_v0_par_essai = df[df['V'] == 0].groupby('essai')['P1_reel'].first()
df_ut['P1_reel'] = df['essai'].map(p1_v0_par_essai).fillna(df['P1_reel'])

lq_valide_par_essai = df.dropna(subset=['Lq_reel']).groupby('essai')['Lq_reel'].first()
df_ut['Lq_reel'] = df['essai'].map(lq_valide_par_essai).fillna(df['Lq_reel'])

df_ut = df_ut.dropna()  # Supprimer les lignes avec des valeurs manquantes
X1 = df_ut.drop(columns=['E2.1','E2.2','E2.3','E2.4'])
y1 = df_ut[['E2.1', 'E2.2', 'E2.3', 'E2.4']]
y2 = y1.mean(axis=1)  # Moyenne des colonnes E2.1, E2.2, E2.3, E2.4
print(X1)
X_train, X_test, y_train, y_test = train_test_split(X1, y2, test_size=0.2, random_state=42)


       P1_reel   Lq_reel        P4         V        E2  P6_reel        E1
4     0.531993  0.139073  0.646018  0.000000  0.750538     3.20  0.535429
5     0.531993  0.139073  0.646018  0.000000  0.597849     3.20  0.486157
6     0.531993  0.139073  0.646018  0.000000  0.604301     3.20  0.494603
7     0.531993  0.139073  0.646018  0.000000  0.616487     3.20  0.481933
12    0.531993  0.139073  0.646018  0.000000  0.764875     4.92  0.512905
...        ...       ...       ...       ...       ...      ...       ...
2799  0.698355  0.635762  1.000000  0.990033  0.635125     5.12  0.324261
2812  0.957952  0.685430  1.000000  0.989369  0.827240     5.12  0.381980
2813  0.957952  0.685430  1.000000  0.989369  0.809319     5.12  0.383388
2814  0.957952  0.685430  1.000000  0.989369  0.980645     5.12  0.312999
2815  0.957952  0.685430  1.000000  0.989369  0.670968     5.12  0.308775

[1107 rows x 7 columns]


/var/folders/db/0jvml7bx2wj5gv9322q7lczc0000gn/T/ipykernel_69595/546814884.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ut['P1_reel'] = df['essai'].map(p1_v0_par_essai).fillna(df['P1_reel'])
/var/folders/db/0jvml7bx2wj5gv9322q7lczc0000gn/T/ipykernel_69595/546814884.py:10: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_ut['Lq_reel'] = df['essai'].map(lq_valide_par_essai).fillna(df['Lq_reel'])


In [4]:
param_grid = {
    'n_estimators': [100, 200, 600, 800],      # On cherche le bon nombre d'arbres
    'max_depth': [5, 8, 12, None],             # On cherche la bonne profondeur (PAS DE 'None' !)
    'min_samples_leaf': [2, 3, 5]        # On cherche à lisser plus ou moins les feuilles
}

# Modèle de base pour la recherche
rf_base = RandomForestRegressor(random_state=42)

print("Recherche des meilleurs paramètres pour E2_moy en cours...")
# cv=5 : Validation croisée en 5 blocs. n_jobs=-1 : Utilise tout le CPU.
grid_moy = GridSearchCV(estimator=rf_base, param_grid=param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1)
grid_moy.fit(X_train, y_train)
rf_moy = grid_moy.best_estimator_
print(f"Meilleurs paramètres E2_moy : {grid_moy.best_params_}")

print("--- 4. Évaluation ---")
# Prédictions (avec les meilleurs modèles trouvés automatiquement)
pred_moy = rf_moy.predict(X_test)

# Métriques pour E2_moy
print(">> PERFORMANCES SUR E2_moy (La valeur moyenne) :")
print(f"MAE  : {mean_absolute_error(y_test, pred_moy):.4f}")
print(f"RMSE : {np.sqrt(mean_squared_error(y_test, pred_moy)):.4f}")
print(f"R2   : {r2_score(y_test, pred_moy):.4f}\n")

Recherche des meilleurs paramètres pour E2_moy en cours...
Meilleurs paramètres E2_moy : {'max_depth': 12, 'min_samples_leaf': 2, 'n_estimators': 100}
--- 4. Évaluation ---
>> PERFORMANCES SUR E2_moy (La valeur moyenne) :
MAE  : 0.0504
RMSE : 0.0748
R2   : 0.8107



In [5]:
# Modèle Random Forest
rf = RandomForestRegressor(
    n_estimators=200,   # nombre d'arbres
    max_depth= 8,    # profondeur illimitée
    random_state=42,
    n_jobs=-1          # utilise tous les cœurs CPU
)

# Entraînement
rf.fit(X_train, y_train)

# Prédictions
y_pred = rf.predict(X_test)
y_pred = pd.Series(y_pred, index=y_test.index)

# Métriques
r2 = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)

print(f"R2: {r2:.06f}")
print(f"MSE: {mse:.06f}")
print()

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse) # La racine carrée pour retrouver l'unité physique de E1

print(f"Erreur Absolue Moyenne (MAE) : {mae:.2f}")
print(f"Racine de l'Erreur Quadratique Moyenne (RMSE) : {rmse:.2f} ")
print("\n")

print("--- 4. Importance des Paramètres ---")
# On récupère le classement d'importance des variables calculé par le modèle
importances = rf.feature_importances_
for col, imp in sorted(zip(X1.columns, importances), key=lambda x: x[1], reverse=True):
    print(f"{col:<10}: {imp*100:.1f}%")


R2: 0.806569
MSE: 0.005718

Erreur Absolue Moyenne (MAE) : 0.05
Racine de l'Erreur Quadratique Moyenne (RMSE) : 0.08 


--- 4. Importance des Paramètres ---
E1        : 44.1%
P4        : 21.2%
Lq_reel   : 13.0%
E2        : 7.8%
V         : 7.2%
P6_reel   : 4.9%
P1_reel   : 1.8%
